# 1C–1F — Comparación de encoders en un solo notebook

Ejecuta **de una** los cuatro encoders del track inglés (BioBERT, SciBERT,
BiomedBERT-large, BioLinkBERT-large) con **config idéntica** a 1A/1B
(`lr=2e-5, max_len=256, warmup=300, neg_ratio=3, seed=42, 15 epochs`), guarda
todo en la carpeta de cada uno y al final saca la tabla comparativa con 1A y 1B.

- Los **large** usan `batch=4` (11.5 GB VRAM); los **base**, `batch=16`.
- Entre modelo y modelo se libera la GPU (`empty_cache`).
- **Aviso:** es un run largo (varias horas, sobre todo los large). Déjalo corriendo.
- Cada carpeta queda autocontenida (checkpoint + métricas + artefactos), así que
  si se corta, los modelos ya terminados no se reentrenan: puedes comentar en
  `CONFIGS` los que ya estén hechos y relanzar.

## 1. Setup

In [ ]:
# Ejecucion en servidor local (zape), entorno conda "tfg".
# Override de la cache de HuggingFace a una carpeta escribible del home.
# EJECUTAR ANTES de cualquier import de opennre/transformers/huggingface_hub.
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    "Reinicia el kernel y ejecuta esta celda ANTES de cualquier import de HF/opennre.")
print("HF cache:", os.environ["HF_HUB_CACHE"])

# Parches de compatibilidad de OpenNRE (UTF-8, AdamW de torch, num_workers=0)
!python ../baseline/patch_opennre.py

In [ ]:
import json, importlib, time, logging, gc
from collections import Counter
from pathlib import Path
import nltk, pandas as pd, torch, numpy as np

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

## 2. Configuración global y lista de encoders

In [ ]:
# Hiperparametros comunes (identicos a 1A/1B)
MAX_LENGTH   = 256
LEARNING_RATE= 2e-5
EPOCHS       = 15
WARMUP_STEPS = 300
SEED         = 42
NEG_RATIO    = 3

# Datos (formato OpenNRE ya preparado)
DATA_DIR   = Path("../data/english")
TRAIN_DATA = DATA_DIR / "eng_train.txt"
DEV_DATA   = DATA_DIR / "eng_dev.txt"
REL2ID_PATH= DATA_DIR / "rel2id.json"
for p in (TRAIN_DATA, DEV_DATA, REL2ID_PATH):
    assert p.exists(), f"FALTA {p}"
with open(REL2ID_PATH) as f:
    rel2id = json.load(f)
print("Clases:", len(rel2id))

# Los 4 encoders a comparar. Comenta los que ya tengas hechos para no repetir.
CONFIGS = [
    {"exp": "biobert",           "model": "dmis-lab/biobert-v1.1",                                  "batch": 16, "outdir": "1C-biobert"},
    {"exp": "scibert",           "model": "allenai/scibert_scivocab_uncased",                       "batch": 16, "outdir": "1D-scibert"},
    {"exp": "biomedbert_large",  "model": "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract",  "batch": 4,  "outdir": "1E-biomedbert-large"},
    {"exp": "biolinkbert_large", "model": "michiyasunaga/BioLinkBERT-large",                        "batch": 4,  "outdir": "1F-biolinkbert-large"},
]
for c in CONFIGS:
    print(f"  {c['exp']:<20} {c['model']:<55} batch={c['batch']}")

## 3. Parche macro_f1 y bucle de entrenamiento

In [ ]:
import sys
sys.path.insert(0, "../baseline")
from patch_opennre import add_macro_f1_metric
add_macro_f1_metric()

from opennre.framework.utils import AverageMeter
from tqdm import tqdm

def train_with_history(fw, max_epoch, metric="macro_f1"):
    """Reimplementa el bucle de SentenceRE para validar y guardar el mejor
    checkpoint por macro_f1 en cada epoch, devolviendo el historial."""
    history, best_metric = [], 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss, avg_acc = AverageMeter(), AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try: data[i] = data[i].cuda()
                    except Exception: pass
            label, args = data[0], data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1); avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward(); fw.optimizer.step()
            if fw.scheduler is not None: fw.scheduler.step()
            fw.optimizer.zero_grad()
        val = fw.eval_model(fw.val_loader)
        rec = {"epoch": epoch, "train_loss": avg_loss.avg, "train_acc": avg_acc.avg,
               "val_acc": val["acc"], "val_micro_p": val["micro_p"], "val_micro_r": val["micro_r"],
               "val_micro_f1": val["micro_f1"], "val_macro_f1": val["macro_f1"]}
        history.append(rec)
        print(f"Epoch {epoch}: loss={rec['train_loss']:.4f} "
              f"val_micro_f1={rec['val_micro_f1']:.4f} val_macro_f1={rec['val_macro_f1']:.4f}")
        if val[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val[metric]:.4f}, guardando checkpoint")
            folder = "/".join(fw.ckpt.split("/")[:-1])
            if folder and not os.path.exists(folder): os.makedirs(folder, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val[metric]
    print(f"Mejor {metric} en val: {best_metric:.4f}")
    return history

## 4. Función que ejecuta un encoder completo

In [ ]:
from save_artifacts import dump_all_artifacts

# dev se carga una vez (igual para todos)
dev_instances = [json.loads(l) for l in open(DEV_DATA, encoding="utf-8") if l.strip()]
all_relations = sorted(rel2id.keys())

def _metrics(rows):
    gold = [i["relation"] for i in dev_instances]
    pred = [r["relation"] for r in rows]
    total = len(gold); acc = sum(g==p for g,p in zip(gold,pred))/total
    gc_, pc_, tp_ = Counter(gold), Counter(pred), Counter()
    for g,p in zip(gold,pred):
        if g==p: tp_[g]+=1
    per, f1s = {}, []
    for rel in all_relations:
        tp=tp_.get(rel,0); pt=pc_.get(rel,0); gt=gc_.get(rel,0)
        P=tp/pt if pt else 0; R=tp/gt if gt else 0; F=2*P*R/(P+R) if (P+R) else 0
        per[rel]={"precision":P,"recall":R,"f1":F,"support":gt}
        if gt>0: f1s.append(F)
    return acc, (sum(f1s)/len(f1s) if f1s else 0), per

def run_one_encoder(cfg):
    EXP, MODEL, BATCH = cfg["exp"], cfg["model"], cfg["batch"]
    OUT = Path(f"../outputs/{cfg['outdir']}"); OUT.mkdir(parents=True, exist_ok=True)
    CKPT = OUT / f"eng_{EXP}.pth.tar"; PRED = OUT / f"eng_pred_{EXP}.tsv"
    print("\n" + "#"*70 + f"\n# {EXP}  |  {MODEL}  |  batch={BATCH}\n" + "#"*70)

    encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL)
    model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
    framework = opennre.framework.SentenceRE(
        model=model, train_path=str(TRAIN_DATA), val_path=str(DEV_DATA), test_path=str(DEV_DATA),
        ckpt=str(CKPT), batch_size=BATCH, max_epoch=EPOCHS, lr=LEARNING_RATE,
        opt="adamw", warmup_step=WARMUP_STEPS)
    n_params = sum(p.numel() for p in model.parameters())

    t0 = time.time()
    history = train_with_history(framework, EPOCHS, metric="macro_f1")
    minutes = (time.time()-t0)/60
    with open(OUT / f"history_{EXP}.json", "w") as f: json.dump(history, f, indent=2)

    # recargar el MEJOR checkpoint sobre el mismo modelo (sin duplicar memoria)
    model.load_state_dict(torch.load(str(CKPT), map_location="cpu")["state_dict"])
    if torch.cuda.is_available(): model = model.cuda()
    model.eval()

    # prediccion en dev
    rows = []
    for inst in dev_instances:
        pred_rel, score = model.infer({"text": inst["text"],
            "h": {"pos": inst["h"]["pos"]}, "t": {"pos": inst["t"]["pos"]}})
        rows.append({"document_id": inst["doc_id"], "relation": pred_rel, "score": score,
            "gold": inst["relation"], "head_text": inst["h"]["name"], "head_span": inst["head_span"],
            "head_type": inst["head_type"], "tail_text": inst["t"]["name"],
            "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]})
    pred_df = pd.DataFrame(rows)
    exp_df = pred_df[["document_id","relation","head_text","head_span","head_type",
                      "tail_text","tail_span","tail_type"]]
    exp_df[exp_df["relation"]!="no_relation"].to_csv(PRED, sep="\t", index=False)

    acc, macro_f1, per = _metrics(rows)
    best = max(history, key=lambda h: h["val_macro_f1"])
    results = {"experiment": EXP, "model": MODEL,
        "hyperparameters": {"max_length":MAX_LENGTH,"batch_size":BATCH,"learning_rate":LEARNING_RATE,
            "epochs":EPOCHS,"warmup_steps":WARMUP_STEPS,"neg_ratio":NEG_RATIO,"seed":SEED},
        "results": {"accuracy":acc, "macro_f1":macro_f1, "micro_f1":best["val_micro_f1"],
            "best_epoch":best["epoch"], "per_relation":per},
        "n_params": int(n_params), "train_minutes": round(minutes,1)}
    with open(OUT / f"results_{EXP}.json", "w") as f: json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"[{EXP}] Macro F1 dev = {macro_f1:.4f} | micro = {best['val_micro_f1']:.4f} | {minutes:.1f} min")

    # artefactos extra (run_meta, dev_probs/gold, preds detalladas, confusion.csv, bootstrap CI)
    hparams = results["hyperparameters"]
    try:
        dump_all_artifacts(model, encoder, dev_instances, pred_df, rel2id,
                           OUT, EXP, MODEL, hparams, history=history, batch_size=BATCH)
    except Exception as e:
        import traceback; traceback.print_exc(); print("AVISO artefactos:", e)

    summary = {"exp":EXP, "model":MODEL, "macro_f1":macro_f1, "micro_f1":best["val_micro_f1"],
               "accuracy":acc, "best_epoch":best["epoch"], "n_params":int(n_params), "minutes":round(minutes,1)}
    # liberar GPU antes del siguiente modelo
    del framework, model, encoder; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary

## 5. Ejecutar los 4 encoders

In [ ]:
summaries = []
for cfg in CONFIGS:
    OUT = Path(f"../outputs/{cfg['outdir']}")
    done = (OUT / f"results_{cfg['exp']}.json")
    if done.exists():
        print(f"[skip] {cfg['exp']} ya tiene results_*.json -> lo cargo sin reentrenar")
        summaries.append({**json.load(open(done))["results"], "exp":cfg["exp"], "model":cfg["model"]})
        continue
    summaries.append(run_one_encoder(cfg))

print("\nTODOS LOS ENCODERS COMPLETADOS")

## 6. Tabla comparativa final (con 1A y 1B)

In [ ]:
# Reune 1A, 1B y los 4 nuevos desde sus results_*.json
def load_result(path, exp):
    p = Path(path)
    if not p.exists(): return None
    d = json.load(open(p))["results"]
    return {"exp": exp, "macro_f1": d["macro_f1"], "micro_f1": d.get("micro_f1"),
            "best_epoch": d.get("best_epoch")}

rows = [
    load_result("../outputs/1A-pubmedbert/results_pubmedbert.json", "1A PubMedBERT"),
    load_result("../outputs/1B-biolinkbert/results_biolinkbert_base.json", "1B BioLinkBERT-base"),
]
for cfg in CONFIGS:
    rows.append(load_result(f"../outputs/{cfg['outdir']}/results_{cfg['exp']}.json",
                            f"{cfg['outdir']}"))
rows = [r for r in rows if r]
tab = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
tab["diff_vs_baseline"] = tab["macro_f1"] - 0.6944
print(tab.to_string(index=False))

Path("../outputs").mkdir(exist_ok=True)
tab.to_csv("../outputs/comparacion_encoders.csv", index=False)
print("\nGuardado: ../outputs/comparacion_encoders.csv")